### import libraries

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.multioutput import MultiOutputClassifier
from sklearn import tree
import random
from PIL import Image
import cv2
import opendatasets as od 

### download dataset

In [2]:
# Uncomment to download the dataset
# od.download(
#     "https://www.kaggle.com/datasets/andrewmvd/ocular-disease-recognition-odir5k" ,force=True)

In [2]:
df = pd.read_excel("../Processed_data.xlsx")
df.head(10)

,Patient ID,Patient Age,Patient Sex,Filename,Diagnosis,N,D,G,C,A,H,M,O
0,0,69,Female,0_left.jpg,cataract,0,0,0,1,0,0,0,0
1,1,57,Male,1_left.jpg,normal fundus,1,0,0,0,0,0,0,0
2,2,42,Male,2_left.jpg,"laser spot,moderate non proliferative retinopathy",0,1,0,0,0,0,0,1
3,3,66,Male,3_left.jpg,normal fundus,1,0,0,0,0,0,0,0
4,4,53,Male,4_left.jpg,macular epiretinal membrane,0,0,0,0,0,0,0,1
5,5,50,Female,5_left.jpg,moderate non proliferative retinopathy,0,1,0,0,0,0,0,0
6,6,60,Male,6_left.jpg,macular epiretinal membrane,0,0,0,0,0,0,0,1
7,7,60,Female,7_left.jpg,drusen,0,0,0,0,0,0,0,1
8,8,59,Male,8_left.jpg,normal fundus,1,0,0,0,0,0,0,0
9,9,54,Male,9_left.jpg,normal fundus,1,0,0,0,0,0,0,0


### modifying the dataframe

In [3]:
# Function to create the target Label for the decision tree
def createTarget(df):
    # Create a new column 'Labels' in the DataFrame and initialize it with a list of 8 zeros for each row
    df['Labels'] = [[0, 0, 0, 0, 0, 0, 0, 0] for i in range(0, len(df))]

    # Define the target columns you want to extract values from
    target_columns = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']

    # Create an empty list called 'label' to store values for each row
    label = []

    # Loop through each row in the DataFrame
    for i in range(0, len(df)):
        # Loop through the target columns
        for target in target_columns:
            # Append the value from the current target column for the current row to the 'label' list
            label.append(df.loc[i, target])

        # Update the 'Labels' column for the current row with the 'label' list
        df.at[i, 'Labels'] = label

        # Reset the 'label' list for the next row
        label = []

In [4]:
# Call the 'createTarget' function to modify the 'df' DataFrame
createTarget(df)
df.head(10)

,Patient ID,Patient Age,Patient Sex,Filename,Diagnosis,N,D,G,C,A,H,M,O,Labels
0,0,69,Female,0_left.jpg,cataract,0,0,0,1,0,0,0,0,"[0, 0, 0, 1, 0, 0, 0, 0]"
1,1,57,Male,1_left.jpg,normal fundus,1,0,0,0,0,0,0,0,"[1, 0, 0, 0, 0, 0, 0, 0]"
2,2,42,Male,2_left.jpg,"laser spot,moderate non proliferative retinopathy",0,1,0,0,0,0,0,1,"[0, 1, 0, 0, 0, 0, 0, 1]"
3,3,66,Male,3_left.jpg,normal fundus,1,0,0,0,0,0,0,0,"[1, 0, 0, 0, 0, 0, 0, 0]"
4,4,53,Male,4_left.jpg,macular epiretinal membrane,0,0,0,0,0,0,0,1,"[0, 0, 0, 0, 0, 0, 0, 1]"
5,5,50,Female,5_left.jpg,moderate non proliferative retinopathy,0,1,0,0,0,0,0,0,"[0, 1, 0, 0, 0, 0, 0, 0]"
6,6,60,Male,6_left.jpg,macular epiretinal membrane,0,0,0,0,0,0,0,1,"[0, 0, 0, 0, 0, 0, 0, 1]"
7,7,60,Female,7_left.jpg,drusen,0,0,0,0,0,0,0,1,"[0, 0, 0, 0, 0, 0, 0, 1]"
8,8,59,Male,8_left.jpg,normal fundus,1,0,0,0,0,0,0,0,"[1, 0, 0, 0, 0, 0, 0, 0]"
9,9,54,Male,9_left.jpg,normal fundus,1,0,0,0,0,0,0,0,"[1, 0, 0, 0, 0, 0, 0, 0]"


In [5]:
# Create a new DataFrame 'upd_df' by dropping specified columns from the original DataFrame 'df'
# The columns 'Patient ID', 'Patient Age', 'Patient Sex', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O', 'Diagnosis'
# are removed from 'upd_df'
upd_df = df.drop(['Patient ID', 'Patient Sex', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O', 'Diagnosis'], axis=1)

# 'upd_df' now contains the remaining columns after dropping the specified ones
upd_df.head(5)

,Patient Age,Filename,Labels
0,69,0_left.jpg,"[0, 0, 0, 1, 0, 0, 0, 0]"
1,57,1_left.jpg,"[1, 0, 0, 0, 0, 0, 0, 0]"
2,42,2_left.jpg,"[0, 1, 0, 0, 0, 0, 0, 1]"
3,66,3_left.jpg,"[1, 0, 0, 0, 0, 0, 0, 0]"
4,53,4_left.jpg,"[0, 0, 0, 0, 0, 0, 0, 1]"


In [6]:
#Min Max Scaling Age
age_scaler = MinMaxScaler()
upd_df['Patient Age'] = age_scaler.fit_transform(upd_df[['Patient Age']])

In [7]:
upd_df.head()

,Patient Age,Filename,Labels
0,0.755556,0_left.jpg,"[0, 0, 0, 1, 0, 0, 0, 0]"
1,0.622222,1_left.jpg,"[1, 0, 0, 0, 0, 0, 0, 0]"
2,0.455556,2_left.jpg,"[0, 1, 0, 0, 0, 0, 0, 1]"
3,0.722222,3_left.jpg,"[1, 0, 0, 0, 0, 0, 0, 0]"
4,0.577778,4_left.jpg,"[0, 0, 0, 0, 0, 0, 0, 1]"


In [8]:
# Path to your image dataset
image_dataset_path = "../ocular-disease-recognition-odir5k/ODIR-5K/ODIR-5K/Training images"

### creating arrays to hold the image features 

In [10]:
# Initialize the lists to hold the feature vectors and labels
features = []
labels = []

# Process the images and labels:
for index, row in upd_df.iterrows():
    # Open the image
    img_path = os.path.join(image_dataset_path, row['Filename'])
    with Image.open(img_path) as img:
        
        img = img.resize((128, 128))  #resizing 
        img_array = np.array(img)  # Convert to numpy array
        img_array = img_array.flatten()  # Flatten the image array
    
    # Append the age to the image array.
    patient_age = np.array(row['Patient Age'])
    patient_age = np.expand_dims(patient_age, axis=0)
    feature_with_age = np.concatenate((img_array, patient_age)) #concatenating both features

    # Append the feature_with_age array to the features list
    features.append(feature_with_age)
    
    # Append the label (which is already a list) to the labels list
    labels.append(row['Labels'])

# Convert lists to numpy arrays
features = np.array(features)
labels = np.array(labels)



In [11]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.3, random_state=42)

# Initialize the Decision Tree Classifier
clf = MultiOutputClassifier(DecisionTreeClassifier(random_state=42))

# Train the model
clf.fit(X_train, y_train)


# Predict the labels on the test set
y_pred = clf.predict(X_test)


In [12]:
from sklearn.metrics import classification_report

In [16]:
# Calculate the accuracy of a machine learning model by comparing its predictions 'y_pred'
# to the true labels 'y_test' from the test set
accuracy = accuracy_score(y_test, y_pred)

# Print the accuracy of the Decision Tree classifier on the test set as a percentage with two decimal places
print(f'Accuracy of the Decision Tree classifier on the test set: {accuracy:.2%}')

Accuracy of the Decision Tree classifier on the test set: 17.57%


In [14]:
classification_report = classification_report(y_pred, y_test)

C:\Users\mimiw\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [15]:
print(classification_report)

              precision    recall  f1-score   support

           0       0.52      0.49      0.51      1019
           1       0.31      0.30      0.30       556
           2       0.16      0.15      0.15       108
           3       0.24      0.19      0.21       128
           4       0.09      0.08      0.08       103
           5       0.09      0.08      0.08        76
           6       0.47      0.40      0.43        95
           7       0.20      0.18      0.19       407

   micro avg       0.36      0.34      0.35      2492
   macro avg       0.26      0.23      0.25      2492
weighted avg       0.36      0.34      0.35      2492
 samples avg       0.38      0.28      0.31      2492



 
#### Analysis

Using decision Tree with paramters (age & images) to predict the diagnosis gives a higher accuracy percentage (17.57%) in comparison to Using decision Tree with only images  (15.81%).
This suggests that using only the raw images provided a certain level of predictive ability but is relatively low, indicating that the model might struggle to capture the complexity of the images or that the features are not sufficiently informative. 

The addition of age appears to have improved accuracy slightly, but overall the accuracy is rather poor. 
This is typical of underfitting, and suggests the model was unable to capture the higher dimensional features of the image.



